# Setup
- Imports
- Define Helper Function. 
- Define and show relevent paths
- Get FFMPEG Dependency by Disecting bundled package imagio (docker eat your heartout).

In [7]:
import os
import pandas as pd
import shutil
import tempfile
import subprocess
from pathlib import Path
import imageio_ffmpeg
import whisper
import whisper.audio
import time
import re
import json

In [8]:
# Configure Whisper to use imageio-ffmpeg directly (Docker Eat your heart out!)
ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()
whisper.audio.FFMPEG = ffmpeg_path

print("FFmpeg version:", imageio_ffmpeg.get_ffmpeg_version())

DOWNLOAD_FOLDER = os.path.normpath(
    os.path.join("..","..", "Data", "Landing_Zone", "Recordings")
)
BASE_FOLDER = Path(DOWNLOAD_FOLDER)
TRANSCRIPT_FOLDER = Path("../../Data/Processed/transcripts")
TRANSCRIPT_FOLDER.mkdir(parents=True, exist_ok=True)
MODEL_NAME = "tiny" # Wisper Model to use for transcription. Options: tiny, base, small, medium, large. 

print("DOWNLOAD_FOLDER:", os.path.abspath(DOWNLOAD_FOLDER))
print("Base folder:", os.path.abspath(BASE_FOLDER))
print("Transcript folder:", os.path.abspath(TRANSCRIPT_FOLDER))
print("Number of audio files found:", len(list(BASE_FOLDER.glob("*.opus"))))

FFmpeg version: 9.0.2-essentials_build-www.gyan.dev
DOWNLOAD_FOLDER: c:\Users\Graem\Desktop\Github Repos\EMF_Camp_DataEng\Data\Landing_Zone\Recordings
Base folder: c:\Users\Graem\Desktop\Github Repos\EMF_Camp_DataEng\Data\Landing_Zone\Recordings
Transcript folder: c:\Users\Graem\Desktop\Github Repos\EMF_Camp_DataEng\Data\Processed\transcripts
Number of audio files found: 130


#  Derive Transcripts from downloaded .opeus

In [10]:
# Identify recordings that still need transcription
audio_files = [
    path for path in BASE_FOLDER.iterdir()
    if path.suffix.lower() in {".mp3", ".opus"}
]
total_audio_files = len(audio_files)
transcripts_to_process = [
    path for path in audio_files
    if not (
        TRANSCRIPT_FOLDER
        / f"{path.name}.txt"
    ).exists()
]

if not transcripts_to_process:
    print("All recordings have already been transcribed.")

elif total_audio_files == 0:
    print("No audio files found. Check download folder.")

else:
    print(f"{len(transcripts_to_process)} recordings need transcription.")

    # Load Whisper only when there is work to do
    print(f"Loading Whisper model: {MODEL_NAME}")
    model = whisper.load_model(MODEL_NAME)

    start_time = time.time()
    processed_times = []

    for index, audio_file in enumerate(transcripts_to_process, 1):
        transcript_title = re.sub(r"\.(?:opus|mp3)$", "", audio_file.name, flags=re.IGNORECASE) # re Removes file extension
        output_file = TRANSCRIPT_FOLDER / f"{transcript_title}.txt"

        # Skip files already transcribed
        if output_file.exists():
            print(f"[{index}/{total_audio_files}] Skipping: {audio_file.name}")
            continue

        print(f"\n[{index}/{total_audio_files}] Transcribing: {audio_file.name} \n to location: {os.path.abspath(output_file)}")
        file_start = time.time()

        try:
            # Transcribe using Whisper and the configured FFmpeg executable
            result = model.transcribe(
                str(audio_file),
                language="en",
                fp16=False,
                verbose=False,
            )

            # Save timestamped transcript
            with output_file.open("w", encoding="utf-8") as f:
                for segment in result["segments"]:
                    start = segment["start"]
                    text = segment["text"].strip()

                    minutes = int(start // 60)
                    seconds = int(start % 60)

                    f.write(f"[{minutes:02d}:{seconds:02d}] {text}\n")

            # Calculate processing time
            elapsed = time.time() - file_start
            processed_times.append(elapsed)

            # Estimate remaining time
            remaining_files = sum(
                1 for remaining_file in audio_files[index:]
                if not (
                    TRANSCRIPT_FOLDER /
                    f"{extract_title(remaining_file.name, safe_filename=True)}.txt"
                ).exists()
            )

            average_time = sum(processed_times) / len(processed_times)
            remaining_seconds = remaining_files * average_time
            minutes, seconds = divmod(int(remaining_seconds), 60)

            print(f"Saved: {output_file.name}")
            print(
                f"Progress: {index}/{total_files} | "
                f"Elapsed: {(time.time() - start_time)/60:.1f} min | "
                f"ETA: {minutes}m {seconds}s"
            )

        except Exception as error:
            print(f"ERROR processing {audio_file.name}: {error}")

    print("\nAll transcription jobs completed.") #  Run on my GPUless laptop for all files took on tiny 304.3 min

130 recordings need transcription.
Loading Whisper model: tiny
[1/130] Skipping: 17419 - Apollo 11 - Q&A With Stephen Slater and Helen O'Hara.opus
[2/130] Skipping: 17420 - Opening Ceremony.opus
[3/130] Skipping: 17422 - Rotundest Robin： Building a Smart Bird Feeder That Weighs Songbirds.opus
[4/130] Skipping: 17424 - Boiling A Frog： Science & Ethics as TTRPG.opus
[5/130] Skipping: 17425 - Hacking APRS to warm my camper van whilst I’m still in the pub.opus
[6/130] Skipping: 17426 - The Hassle Hurdle： a philosophy to hack ADHD.opus
[7/130] Skipping: 17427 - An Appreciation of Ecological Sanitation (or Turds for Nerds).opus
[8/130] Skipping: 17428 - Sonus ex nihilo.opus
[9/130] Skipping: 17429 - SMolSTM： an open hardware scanning tunnelling microscope for creating single-molecule circuits.opus
[10/130] Skipping: 17430 - Middle Rage - Social Media and the War on Democracy.opus
[11/130] Skipping: 17431 - Mountains, Rescues, and Risks.opus
[12/130] Skipping: 17432 - Project CETI： How to Dec

# Build Transcripts datafame 

### Helper function

In [20]:
def extract_title(name, *, safe_filename=False, normalize=False):
    name = re.sub(r"\.(?:opus|mp3|txt)$", "", name, flags=re.IGNORECASE)  # Remove file extension
    name = re.sub(r"^\d+\s*-\s*", "", name).strip()                       # Remove leading numbers and dashes, and strip whitespace
    if safe_filename:
        name = re.sub(r'[<>:"/\\|?*]', "_", name).strip(" .")             # [...] any one of these characters (illegal in filenames) -> "_"
    if normalize:
        name = re.sub(r"[\W_]+", " ", name.casefold()).strip()            # lowercases flexible to diffrent languages with casefold, replaces _ with spaces thereby helping to normalise for fuzzy matching. 
    return name

### Link transcripts file to the source URL for the website 
(Could replace later with a downloaded media approach. )

Test me 

In [12]:
METADATA_PATH = Path("../../Data/Landing_Zone/recording_metadata.json")

# Load recording metadata
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

# Extract ID, title and URL from metadata json dump. 
recordings_df = pd.DataFrame([
    {
        "id": str(entry.get("id", "")).strip(),
        "recording_title": entry.get("title"),
        "url": entry.get("webpage_url") or entry.get("original_url"),
    }
    for entry in metadata.get("entries", [])
    if entry
])

# Build transcript DataFrame
transcript_df = pd.DataFrame([
    {
        "id": path.stem.split(" - ", 1)[0].strip(),
        "title": extract_title(path.name, safe_filename=True),
        "transcript": path.read_text(encoding="utf-8"),
    }
    for path in TRANSCRIPT_FOLDER.glob("*.txt") # get all with pattern "*.txt"
])

# Link transcripts to recording metadata using ID
transcript_df = transcript_df.merge(
    recordings_df[["id", "url"]],
    on="id",
    how="left"
)

# Check matching results
matched_transcripts = transcript_df["url"].notna().sum()
unmatched_transcripts = transcript_df.loc[transcript_df["url"].isna(), ["id", "title"]]
print(f"Recordings: {len(recordings_df)}")
print(f"Transcripts: {len(transcript_df)}")
print(f"Matched: {matched_transcripts}")
print(f"Unmatched: {len(unmatched_transcripts)}")
if not unmatched_transcripts.empty:
    print("\nTranscripts without a matching recording:")
    for _, row in unmatched_transcripts.iterrows():
        print(f"- {row['id']} | {row['title']}")

display(transcript_df.head())

Recordings: 130
Transcripts: 130
Matched: 130
Unmatched: 0


,id,title,transcript,url
0,17419,Apollo 11 - Q&A With Stephen Slater and Helen ...,"[00:00] Hello, everybody. My name is Helen and...",https://media.ccc.de/v/emf2026-364-1-apollo-11...
1,17420,Opening Ceremony.txt,"[00:00] All right, thank you.\n[00:14] Thank y...",https://media.ccc.de/v/emf2026-278-1-opening-c...
2,17422,Rotundest Robin： Building a Smart Bird Feeder ...,"[00:00] Hello, hello. What a treat to be speak...",https://media.ccc.de/v/emf2026-147-1-rotundest...
3,17424,Boiling A Frog： Science & Ethics as TTRPG.txt,[00:00] So my name is Mario. My background is ...,https://media.ccc.de/v/emf2026-125-1-boiling-a...
4,17425,Hacking APRS to warm my camper van whilst I’m ...,[00:00] What is it?\n[00:17] Do we have it?\n[...,https://media.ccc.de/v/emf2026-156-1-hacking-a...


## Link Transcripts to Scedule using Title as key

## Load Scedule

In [23]:
schedule = pd.read_csv("../../Data/Landing_Zone/schedule.csv")
display(schedule.head())

,id,type,names,pronouns,title,description,short_description,video_privacy,is_fave,official_content,slug,link,occurrences,cost,equipment,age_range,drop_in,family_friendly,content_note
0,384,performance,"Field-FX organisers, DJ Deathboy",NaN,Demoscene Livecode Jam,We are going to code exciting visual effects i...,Visual livecoding jam with a DJ,public,False,False,demoscene-livecode-jam,https://www.emfcamp.org/schedule/2026/384-demo...,"[{'occurrence_num': 1, 'start_date': '2026-07-...",NaN,NaN,NaN,NaN,NaN,NaN
1,165,workshop,Zoë Hutber,she/her,Aardman Modelling Workshops (all ages),Make your own Aardman character to take home! ...,Build your own clay model of an Aardman charac...,none,False,True,aardman-modelling-workshops-all-ages,https://www.emfcamp.org/schedule/2026/165-aard...,"[{'occurrence_num': 1, 'start_date': '2026-07-...",£15,NaN,6+,False,True,NaN
2,392,meetup,Mike Whisky,NaN,Mesh Networking Networking,"For anyone interested in meshcore, meshtastic,...",Mesh networking fans meetup,public,False,False,mesh-networking-networking,https://www.emfcamp.org/schedule/2026/392-mesh...,"[{'occurrence_num': 1, 'start_date': '2026-07-...",NaN,NaN,NaN,NaN,NaN,NaN
3,82,talk,James Singleton,he/him,💚 Our Family's Electrotech Journey 🌞🏡🔌⛽️🚘️,"The story of how we went all electric, ditched...","How we went all electric, ditched fossil fuels...",public,False,True,our-familys-electrotech-journey,https://www.emfcamp.org/schedule/2026/82-our-f...,"[{'occurrence_num': 1, 'start_date': '2026-07-...",NaN,NaN,NaN,NaN,False,NaN
4,1,music,2xAA,he/they,2xAA,2xAA harnesses the low-fi power of old games c...,Dance music with Game Boys,public,False,True,2xaa,https://www.emfcamp.org/schedule/2026/1-2xaa,"[{'occurrence_num': 1, 'start_date': '2026-07-...",NaN,NaN,NaN,NaN,NaN,NaN


### Find Exact Title Matches for on title for Scedule and Transcripts. 

In [ ]:
import difflib
import re
import unicodedata
import warnings
import pandas as pd

def clean(title):
    """Comparison-only copy of a title; original schedule/transcript titles stay untouched."""
    if pd.isna(title):
        return ""
    base = extract_title(str(title), safe_filename=False)
    norm = unicodedata.normalize("NFKD", base).casefold()
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", norm)).strip()


def similarity(a, b):
    """String similarity from 0 to 100."""
    return difflib.SequenceMatcher(None, a, b).ratio() * 100 if a and b else 0.0


schedule_clean = schedule["title"].fillna("").map(clean)
transcripts_clean = transcript_df["title"].fillna("").map(clean)

# 1. Exact matches via right join (keeps all transcripts; unmatched schedule side is NaN)
exact_right = pd.merge(
    schedule_clean.rename("_clean").to_frame().assign(_schedule_row=schedule.index),
    transcripts_clean.rename("_clean").to_frame().assign(_transcript_row=transcript_df.index),
    on="_clean",
    how="right",
)

exact_links = (
    exact_right.dropna(subset=["_schedule_row"])
    .assign(_schedule_row=lambda d: d["_schedule_row"].astype(int), similarity_score=100.0, match_method="exact")
    [["_schedule_row", "_transcript_row", "similarity_score", "match_method"]]
)

unmatched_t = exact_right.loc[exact_right["_schedule_row"].isna(), "_transcript_row"].unique()

### Apply fuzzy matching for any unmatched transcripts 
This is to account for minor name changes on ccc. 

In [ ]:
SIMILARITY_THRESHOLD = 90  # % - only used when exact match fails
fuzzy_links, fallback_rows = [], []

#  Fuzzy matches only for transcripts that failed exact matching
for t_row in unmatched_t:
    t_str = transcripts_clean.loc[t_row]
    scores = [(s_row, similarity(s_str, t_str)) for s_row, s_str in schedule_clean.items()]
    passing = [(s_row, round(sc, 2)) for s_row, sc in scores if sc >= SIMILARITY_THRESHOLD]

    if passing:
        fuzzy_links.extend( # extend:
            {"_schedule_row": s_row, "_transcript_row": t_row, "similarity_score": sc, "match_method": "fuzzy"}
            for s_row, sc in passing
        )
    else: # Add to list of inexact matches. 
        best_s, best_sc = max(scores, key=lambda x: x[1])
        fallback_rows.append({
            "transcript_title": transcript_df.loc[t_row, "title"],
            "closest_schedule_title": schedule.loc[best_s, "title"],
            "similarity_score": round(best_sc, 2),
        })

links = pd.concat([exact_links, pd.DataFrame(fuzzy_links)], ignore_index=True)

# 3. Attach linked transcripts back to the full schedule
transcript_details = transcript_df[["title", "transcript", "url"]].rename(columns={"title": "transcript_title"})
schedule_transcripts = (
    schedule.join(links.set_index("_schedule_row"))
    .join(transcript_details, on="_transcript_row")
    .drop(columns=["_transcript_row"])
)

# --- Diagnostics ---
print(schedule_transcripts["match_method"].fillna("unmatched").value_counts().to_dict(), "schedule rows by status") # Get report of what match types occured.
print(f"{links['_transcript_row'].nunique()} of {len(transcript_df)} transcripts matched to the schedule") # Confirm all transcripts matched.

if "fuzzy" in schedule_transcripts["match_method"].values:
    display(schedule_transcripts.loc[schedule_transcripts["match_method"] == "fuzzy", ["title", "transcript_title", "similarity_score"]]) # Display fuzzymatched for review.

if fallback_rows:
    warnings.warn(f"{len(fallback_rows)} transcript(s) matched no schedule row.") # handle case where transcript had no coresponding match. 
    display(pd.DataFrame(fallback_rows))

display(schedule_transcripts)

{'unmatched': 421, 'exact': 130} schedule rows by status
130 of 130 transcripts matched to the schedule


,id,type,names,pronouns,title,description,short_description,video_privacy,is_fave,official_content,...,equipment,age_range,drop_in,family_friendly,content_note,similarity_score,match_method,transcript_title,transcript,url
0,384,performance,"Field-FX organisers, DJ Deathboy",NaN,Demoscene Livecode Jam,We are going to code exciting visual effects i...,Visual livecoding jam with a DJ,public,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,165,workshop,Zoë Hutber,she/her,Aardman Modelling Workshops (all ages),Make your own Aardman character to take home! ...,Build your own clay model of an Aardman charac...,none,False,True,...,NaN,6+,False,True,NaN,NaN,NaN,NaN,NaN,NaN
2,392,meetup,Mike Whisky,NaN,Mesh Networking Networking,"For anyone interested in meshcore, meshtastic,...",Mesh networking fans meetup,public,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,82,talk,James Singleton,he/him,💚 Our Family's Electrotech Journey 🌞🏡🔌⛽️🚘️,"The story of how we went all electric, ditched...","How we went all electric, ditched fossil fuels...",public,False,True,...,NaN,NaN,NaN,False,NaN,100.0,exact,💚 Our Family's Electrotech Journey 🌞🏡🔌⛽️🚘️.txt,"[00:00] Thank you very much, everyone. Yeah, s...",https://media.ccc.de/v/emf2026-82-1-our-family...
4,1,music,2xAA,he/they,2xAA,2xAA harnesses the low-fi power of old games c...,Dance music with Game Boys,public,False,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
545,143,workshop,Betty Ching,she/her,Mastering Canva & Cricut for Sticker Design,Ready to bring your sticker ideas to life? Lea...,Sticker making using Canva and Cricut with BET...,public,False,True,...,laptop/tablet/ phone,all ages,False,True,NaN,NaN,NaN,NaN,NaN,NaN
546,267,workshop,Ash Brockwell,He/They,Painting with Natural Earth Pigments,Natural earth pigments offer an attractive and...,"Make your own natural paint using ochres, clay...",public,False,True,...,"Wear old clothes, consider apron or overall",12+,False,False,NaN,NaN,NaN,NaN,NaN,NaN
547,260,workshop,David Miller,he/they,Build a whole modular synthesiser!,Welcome to the world of modular audio synthesi...,We will be building the Music Thing Workshop S...,public,False,True,...,Bring wired headphones or a portable speaker w...,all,False,True,NaN,NaN,NaN,NaN,NaN,NaN
548,257,workshop,Sophia Davis,she/her,Make Your Own Haku Lei,"Learn how to make your own haku lei, enjoy som...","Learn how to make your own haku lei, enjoy som...",public,False,True,...,NaN,all,False,False,"Discussion of economic issues, mention of wild...",NaN,NaN,NaN,NaN,NaN


# Save To Bronze

In [42]:
import pyarrow
schedule_transcripts.drop(columns = ["similarity_score","match_method","transcript_title"]) 

BRONZE_PATH = os.path.normpath(
    os.path.join("..","..", "Data", "Bronze", "enriched_scedule.parquet")
)
schedule_transcripts.to_parquet(BRONZE_PATH)

ModuleNotFoundError: No module named 'pyarrow'